# Supplementary Figures And Statistical Analyses

This notebook contains SI-only plots and statistical summaries supporting the main text. It intentionally reads precomputed CSV files generated by the Make targets rather than running expensive analyses inside Jupyter.

Relevant upstream commands:

```bash
make FCD_JOBS=32 paper-fcd-mw-calibration
make FCD_JOBS=32 paper-fcd-distances
```


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from matplotlib.ticker import MaxNLocator, PercentFormatter
from rdkit import Chem
from rdkit.Chem import Descriptors

ROOT = Path.cwd()
while not (ROOT / "results").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))

import paper_plot_data_loaders as plot_loaders
importlib.reload(plot_loaders)
from paper_plot_data_loaders import PAPER_SEEDS_10, load_fcd_mw_calibration

RESULTS = ROOT / "results"
FIG_DIR = ROOT / "notebooks" / "figures" / "si"
DATA_DIR = FIG_DIR / "plotting_data"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
PAPER_SEED_SET = set(PAPER_SEEDS_10)

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.labelsize": 8.5,
    "axes.titlesize": 9.5,
    "legend.fontsize": 7.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.edgecolor": "#111111",
    "axes.linewidth": 0.9,
    "xtick.color": "#111111",
    "ytick.color": "#111111",
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

METHOD_COLORS = {
    "base": "#4B5563",
    "no_update": "#4B5563",
    "random_ne": "#A3A3A3",
    "positive": "#EE9B00",
    "topk_finetune": "#CA6702",
    "reinvent_rl": "#BB3E03",
    "ne": "#1A759F",
    "combined": "#005F73",
}
NE_LAMBDA_COLORS = {0.10: "#184E77", 0.25: "#1E6091", 0.50: "#1A759F", 0.75: "#168AAD", 1.00: "#34A0A4"}
MW_BAND_COLORS = {
    "mw_bottom_10pct": "#184E77",
    "mw_bottom_20pct": "#1E6091",
    "mw_20_40pct": "#168AAD",
    "mw_40_60pct": "#52B69A",
    "mw_60_80pct": "#B5E48C",
    "mw_top_20pct": "#EE9B00",
    "mw_top_10pct": "#CA6702",
}
MW_BAND_ORDER = [
    "base_resample",
    "mw_bottom_10pct",
    "mw_bottom_20pct",
    "mw_20_40pct",
    "mw_40_60pct",
    "mw_60_80pct",
    "mw_top_20pct",
    "mw_top_10pct",
]
MW_BAND_LABELS = {
    "base_resample": "Baseline\nresample",
    "mw_bottom_10pct": "Bottom\n10% MW",
    "mw_bottom_20pct": "Bottom\n20% MW",
    "mw_20_40pct": "20-40%\nMW",
    "mw_40_60pct": "40-60%\nMW",
    "mw_60_80pct": "60-80%\nMW",
    "mw_top_20pct": "Top\n20% MW",
    "mw_top_10pct": "Top\n10% MW",
}

def style_ax(ax):
    ax.grid(False)
    ax.tick_params(axis="both", which="major", bottom=True, left=True, length=3.5, width=0.8, color="#111111", labelcolor="#111111")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#111111")
        spine.set_linewidth(0.9)

def save_figure(fig, name: str):
    for suffix in ("png", "svg", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{suffix}", bbox_inches="tight")
    print(FIG_DIR / f"{name}.png")

def label_subplots(fig, labels=None, x=-0.12, y=1.045):
    axes = [ax for ax in fig.axes if ax.get_visible()]
    if labels is None:
        labels = [chr(ord("a") + i) for i in range(len(axes))]
    for ax, label in zip(axes, labels):
        ax.text(x, y, label, transform=ax.transAxes, ha="left", va="bottom", fontweight="bold", fontsize=10, color="#111111", clip_on=False)

def read_sample_smiles(path: Path) -> list[str]:
    frame = pd.read_csv(path)
    if "valid" in frame.columns:
        frame = frame[frame["valid"]]
    column = "canonical_smiles" if "canonical_smiles" in frame.columns else "smiles"
    return frame[column].dropna().astype(str).tolist()

def molecular_weight(smiles: str) -> float | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return float(Descriptors.MolWt(mol))

def load_baseline_mw(results_dir: Path) -> pd.DataFrame:
    rows = []
    for seed_dir in sorted(results_dir.glob("seed_*")):
        sample_path = seed_dir / "base_samples.csv"
        if not sample_path.exists():
            continue
        seed = int(seed_dir.name.removeprefix("seed_"))
        if seed not in PAPER_SEED_SET:
            continue
        for smi in read_sample_smiles(sample_path):
            mw = molecular_weight(smi)
            if mw is not None:
                rows.append({"seed": seed, "smiles": smi, "mw": mw})
    return pd.DataFrame(rows)

def mean_ci(df: pd.DataFrame, group_cols: list[str], value_col: str) -> pd.DataFrame:
    out = df.groupby(group_cols, observed=True)[value_col].agg(mean="mean", std="std", n="count").reset_index()
    out["sem"] = out["std"] / np.sqrt(out["n"].clip(lower=1))
    out["t_critical"] = out["n"].map(lambda n: float(stats.t.ppf(0.975, n - 1)) if n > 1 else np.nan)
    out["ci95"] = out["t_critical"] * out["sem"]
    return out


## SI Figure: Liability SMARTS Substructures

This figure renders the exact SMARTS patterns used for the four liability-erasure objectives in the main text. The table exported by this cell is a traceable copy of the pattern names and SMARTS strings imported from `neon_molgen.scoring.LIABILITY_SMARTS`.


In [ ]:
try:
    from rdkit.Chem import Draw, rdDepictor
except ImportError as exc:
    raise RuntimeError(
        'RDKit Draw could not be imported. Run this SI figure cell in the Jupyter/RDKit '
        'environment, or install the RDKit drawing backend dependencies such as libXrender.'
    ) from exc
from neon_molgen.scoring import LIABILITY_SMARTS
from matplotlib.colors import to_rgb

MAIN_LIABILITY_FAMILIES = {
    'reactive': 'Reactive motifs',
    'chelator': 'Metal-binding motifs',
    'charged_motif': 'Charged motifs',
    'assay_interference': 'Assay-interference motifs',
}

# Representative molecules are used for drawing because rendering SMARTS query
# molecules directly produces query-bond artifacts. The actual SMARTS pattern
# imported from neon_molgen.scoring is still used for matching/highlighting.
REPRESENTATIVE_SMILES = {
    'reactive': {
        'acid_chloride': 'CC(=O)Cl',
        'sulfonyl_chloride': 'CS(=O)(=O)Cl',
        'isocyanate': 'CN=C=O',
        'isothiocyanate': 'CN=C=S',
        'aldehyde': 'CC=O',
        'epoxide': 'CC1CO1',
        'aziridine': 'C1CN1',
        'michael_acceptor': 'C=CC(=O)C',
        'alpha_beta_unsaturated_carbonyl': 'C=CC(=O)O',
        'alpha_beta_unsaturated_nitrile': 'C=CC#N',
        'nitroalkene': 'C=C[N+](=O)[O-]',
        'vinyl_sulfone_sulfoxide': 'C=CS(=O)C',
        'haloacetamide': 'ClCC(=O)N',
        'maleimide': 'O=C1NC(=O)C=C1',
        'alkyl_halide': 'CCCl',
    },
    'chelator': {
        'catechol': 'Oc1ccccc1O',
        'hydroxamic_acid': 'CC(=O)NO',
        'amidoxime': 'CC(=N(O))N',
        'thiol': 'CS',
        'dithiocarbamate': 'N(C(=S)S)C',
        'beta_dicarbonyl': 'CC(=O)CCC(=O)C',
    },
    'charged_motif': {
        'quaternary_ammonium': 'C[N+](C)(C)C',
        'sulfonate': 'CS(=O)(=O)[O-]',
        'phosphonate': 'CP(=O)([O-])[O-]',
        'carboxylate': 'CC(=O)[O-]',
        'zwitterion_proxy': '[NH3+]CC(=O)[O-]',
    },
    'assay_interference': {
        'rhodanine': 'O=C1NC(=S)SC1',
        'quinone': 'O=C1C=CC(=O)C=C1',
        'azo': 'c1ccc(N=Nc2ccccc2)cc1',
        'nitro': 'C[N+](=O)[O-]',
        'thiourea': 'NC(=S)N',
        'polyphenol': 'Oc1cccc(O)c1',
    },
}

def pretty_pattern_name(name: str) -> str:
    label = name.replace('_', ' ').title()
    return label.replace('Alpha Beta', 'alpha,beta')

def highlight_bonds_from_match(mol, query, match):
    bonds = []
    if not match:
        return bonds
    for query_bond in query.GetBonds():
        begin = match[query_bond.GetBeginAtomIdx()]
        end = match[query_bond.GetEndAtomIdx()]
        bond = mol.GetBondBetweenAtoms(begin, end)
        if bond is not None:
            bonds.append(bond.GetIdx())
    return bonds

pattern_rows = []
for family, family_label in MAIN_LIABILITY_FAMILIES.items():
    for pattern_name, smarts in LIABILITY_SMARTS[family].items():
        example_smiles = REPRESENTATIVE_SMILES[family][pattern_name]
        mol = Chem.MolFromSmiles(example_smiles)
        query = Chem.MolFromSmarts(smarts)
        match = mol.GetSubstructMatch(query) if mol is not None and query is not None else tuple()
        pattern_rows.append({
            'family': family,
            'family_label': family_label,
            'pattern': pattern_name,
            'pattern_label': pretty_pattern_name(pattern_name),
            'smarts': smarts,
            'example_smiles': example_smiles,
            'example_matches_smarts': bool(match),
        })

liability_patterns = pd.DataFrame(pattern_rows)
liability_patterns.to_csv(DATA_DIR / 'si_liability_substructure_smarts.csv', index=False)
display(liability_patterns)
if not liability_patterns['example_matches_smarts'].all():
    print('Representative molecules without exact SMARTS match; shown without highlighted atoms:')
    display(liability_patterns[~liability_patterns['example_matches_smarts']])

family_colors = {
    'reactive': '#1A759F',
    'chelator': '#52B69A',
    'charged_motif': '#EE9B00',
    'assay_interference': '#CA6702',
}
n_cols = 4
n_rows_by_family = {
    family: int(np.ceil(len(liability_patterns[liability_patterns['family'] == family]) / n_cols))
    for family in MAIN_LIABILITY_FAMILIES
}
total_rows = sum(n_rows_by_family.values())
fig, axes = plt.subplots(total_rows, n_cols, figsize=(11.4, 1.82 * total_rows))
axes = np.asarray(axes).reshape(total_rows, n_cols)

row_offset = 0
for family, family_label in MAIN_LIABILITY_FAMILIES.items():
    family_frame = liability_patterns[liability_patterns['family'] == family].reset_index(drop=True)
    family_rows = n_rows_by_family[family]
    highlight_color = family_colors[family]
    highlight_rgb = to_rgb(highlight_color)
    for local_row in range(family_rows):
        for col in range(n_cols):
            ax = axes[row_offset + local_row, col]
            ax.set_axis_off()
            idx = local_row * n_cols + col
            if idx >= len(family_frame):
                continue
            row = family_frame.iloc[idx]
            mol = Chem.MolFromSmiles(row['example_smiles'])
            query = Chem.MolFromSmarts(row['smarts'])
            match = mol.GetSubstructMatch(query)
            try:
                rdDepictor.Compute2DCoords(mol)
            except Exception:
                pass
            highlight_atoms = list(match)
            highlight_bonds = highlight_bonds_from_match(mol, query, match)
            title_color = '#111111' if match else '#991B1B'
            image = Draw.MolToImage(
                mol,
                size=(340, 220),
                kekulize=True,
                highlightAtoms=highlight_atoms,
                highlightBonds=highlight_bonds,
                highlightColor=highlight_rgb,
            )
            ax.imshow(image)
            ax.text(0.5, -0.045, row['pattern_label'], transform=ax.transAxes, ha='center', va='top', fontsize=7.7, fontweight='bold', color=title_color)
            ax.text(0.5, -0.18, row['smarts'], transform=ax.transAxes, ha='center', va='top', fontsize=5.8, family='monospace', color='#4B5563', wrap=True)
    axes[row_offset, 0].text(-0.18, 0.60, family_label, transform=axes[row_offset, 0].transAxes, ha='right', va='center', rotation=90, fontweight='bold', fontsize=9.5, color=family_colors[family])
    row_offset += family_rows

fig.suptitle('Representative substructures used for liability-erasure objectives', y=0.995, fontsize=11, fontweight='bold')
fig.subplots_adjust(left=0.09, right=0.995, top=0.97, bottom=0.03, hspace=0.82, wspace=0.08)
save_figure(fig, 'si_liability_substructures')
plt.show()


## SI Figure: Molecular-Weight Calibration For FCD And FDD

This figure provides experiment-local rulers for the distribution-distance axes. We compare molecular-weight-filtered subsets of each baseline generator against the corresponding unfiltered baseline reference. The rows use the same calibration procedure for GuacaMol RNN, GuacaMol Transformer, and the REINVENT prior, so each FCD/FDD axis is anchored to chemically interpretable low- and high-MW shifts from the same generator family.


In [ ]:
CALIBRATION_SOURCES = {
    "GuacaMol RNN": {
        "results_dir": RESULTS / "objectives/guacamol_rnn_chelator_removal",
        "calibration_dir": RESULTS / "objectives/guacamol_rnn_chelator_removal/analysis/fcd_mw_calibration",
    },
    "GuacaMol Transformer": {
        "results_dir": RESULTS / "objectives/guacamol_transformer_chelator_lastblock_final",
        "calibration_dir": RESULTS / "objectives/guacamol_transformer_chelator_lastblock_final/analysis/fcd_mw_calibration",
    },
    "REINVENT prior": {
        "results_dir": RESULTS / "external/reinvent4/chelator_replicates",
        "calibration_dir": RESULTS / "external/reinvent4/chelator_replicates/analysis/fcd_mw_calibration",
    },
}

calibration_data = {}
for generator, paths in CALIBRATION_SOURCES.items():
    baseline_mw = load_baseline_mw(paths["results_dir"])
    try:
        calibration_metrics, calibration_summary = load_fcd_mw_calibration(folder=paths["calibration_dir"])
    except FileNotFoundError as exc:
        calibration_metrics = pd.DataFrame()
        calibration_summary = pd.DataFrame()
        print(f"Missing calibration for {generator}. Run the corresponding paper-fcd-mw-calibration target.")
        print(exc)
    calibration_data[generator] = {
        "baseline_mw": baseline_mw,
        "metrics": calibration_metrics,
        "summary": calibration_summary,
    }
    print(f"{generator}: baseline MW rows={len(baseline_mw):,}, calibration rows={len(calibration_metrics):,}")


In [ ]:
def mw_histogram_arrays(frame: pd.DataFrame, bins: int = 60):
    mw_values = frame["mw"].dropna().to_numpy()
    if len(mw_values) == 0:
        return np.array([]), np.array([]), np.array([]), {}
    counts, edges = np.histogram(mw_values, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    quantiles = frame["mw"].quantile([0.1, 0.2, 0.4, 0.6, 0.8, 0.9]).to_dict()
    return mw_values, counts, centers, quantiles

def mw_span_definitions(mw_values: np.ndarray, quantiles: dict[float, float]):
    return [
        ("mw_bottom_10pct", float(mw_values.min()), quantiles[0.1]),
        ("mw_bottom_20pct", float(mw_values.min()), quantiles[0.2]),
        ("mw_20_40pct", quantiles[0.2], quantiles[0.4]),
        ("mw_40_60pct", quantiles[0.4], quantiles[0.6]),
        ("mw_60_80pct", quantiles[0.6], quantiles[0.8]),
        ("mw_top_20pct", quantiles[0.8], float(mw_values.max())),
        ("mw_top_10pct", quantiles[0.9], float(mw_values.max())),
    ]

def prepare_calibration_metrics(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    out = frame[frame["band"].isin(MW_BAND_ORDER)].copy()
    out["band"] = pd.Categorical(out["band"], MW_BAND_ORDER, ordered=True)
    out["band_label_plot"] = out["band"].astype(str).map(MW_BAND_LABELS)
    return out

def plot_calibration_box(ax, metrics: pd.DataFrame, metric: str, ylabel: str, title: str | None = None):
    if not metrics.empty and metric in metrics.columns:
        palette = ["#9CA3AF" if band == "base_resample" else MW_BAND_COLORS.get(band, "#9CA3AF") for band in MW_BAND_ORDER]
        sns.boxplot(
            data=metrics,
            x="band_label_plot",
            y=metric,
            order=[MW_BAND_LABELS[band] for band in MW_BAND_ORDER],
            palette=palette,
            width=0.55,
            linewidth=0.5,
            fliersize=0,
            ax=ax,
        )
        ax.set_xlabel("")
        ax.set_ylabel(ylabel)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    else:
        ax.text(0.5, 0.5, "Calibration CSV missing", ha="center", va="center")
        ax.set_xticks([])
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, loc="left", fontweight="bold")
    style_ax(ax)

generators = list(CALIBRATION_SOURCES)
fig, axes = plt.subplots(len(generators), 4, figsize=(14.8, 3.05 * len(generators)), constrained_layout=True)
if len(generators) == 1:
    axes = np.asarray([axes])

all_metrics = []
all_baseline_mw = []
for row, generator in enumerate(generators):
    baseline_mw = calibration_data[generator]["baseline_mw"]
    metrics = prepare_calibration_metrics(calibration_data[generator]["metrics"])
    if not metrics.empty:
        all_metrics.append(metrics.assign(generator=generator))
    if not baseline_mw.empty:
        all_baseline_mw.append(baseline_mw[["seed", "mw"]].assign(generator=generator))

    mw_values, counts, centers, quantiles = mw_histogram_arrays(baseline_mw)

    ax = axes[row, 0]
    if len(mw_values) > 0:
        ax.fill_between(centers, counts, step="mid", color="#D1D5DB", alpha=0.75, linewidth=0)
        ax.plot(centers, counts, color=METHOD_COLORS["base"], lw=1.8)
        ax.set_xlabel("Molecular weight" if row == len(generators) - 1 else "")
        ax.set_ylabel(f"{generator}\nNumber of compounds")
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    else:
        ax.text(0.5, 0.5, "No baseline samples", ha="center", va="center")
        ax.set_ylabel(generator)
    if row == 0:
        ax.set_title("Baseline MW distribution", loc="left", fontweight="bold")
    style_ax(ax)

    ax = axes[row, 1]
    if len(mw_values) > 0:
        ax.fill_between(centers, counts, step="mid", color="#E5E7EB", alpha=0.85, linewidth=0)
        ax.plot(centers, counts, color=METHOD_COLORS["base"], lw=1.4)
        for band, low, high in mw_span_definitions(mw_values, quantiles):
            ax.axvspan(low, high, color=MW_BAND_COLORS[band], alpha=0.18, lw=0, label=MW_BAND_LABELS[band].replace("\n", " "))
        ax.set_xlabel("Molecular weight" if row == len(generators) - 1 else "")
        ax.set_ylabel("Number of compounds")
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
        if row == 0:
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(handles, labels, frameon=False, loc="upper right", fontsize=6.2, ncol=1)
    else:
        ax.text(0.5, 0.5, "No baseline samples", ha="center", va="center")
    if row == 0:
        ax.set_title("Highlighted MW subspaces", loc="left", fontweight="bold")
    style_ax(ax)

    plot_calibration_box(
        axes[row, 2],
        metrics,
        "fcd",
        "FCD to unfiltered baseline",
        "FCD scale from MW shifts" if row == 0 else None,
    )
    plot_calibration_box(
        axes[row, 3],
        metrics,
        "fdd",
        "FDD to unfiltered baseline",
        "FDD scale from MW shifts" if row == 0 else None,
    )

label_subplots(fig)
save_figure(fig, "si_fcd_fdd_mw_calibration_all_generators")
plt.show()

if all_metrics:
    pd.concat(all_metrics, ignore_index=True).to_csv(DATA_DIR / "si_fcd_fdd_mw_calibration_metrics_all_generators.csv", index=False)
if all_baseline_mw:
    pd.concat(all_baseline_mw, ignore_index=True).to_csv(DATA_DIR / "si_fcd_fdd_mw_baseline_distribution_all_generators.csv", index=False)


## SI Figure: Actual RNN And Transformer Distribution Shifts

This figure shows the corresponding FCD and FDD values for the actual GuacaMol RNN and Transformer metal-binding-motif-erasure outputs. It complements the MW calibration figure above by showing where the model interventions sit relative to interpretable baseline-subset shifts.


In [ ]:
GUACAMOL_SHIFT_SOURCES = {
    "GuacaMol RNN": {
        "path": RESULTS / "objectives/guacamol_rnn_chelator_removal/analysis/distribution_distance_fcd_base_filtered_selected/distribution_distance_metrics.csv",
        "models": {
            "base": "Base",
            "random_neon_lambda_1.0": "Random NE 1.0",
            "positive": "Positive FT",
            "neon_lambda_1.0": "NE 1.0",
        },
    },
    "GuacaMol Transformer": {
        "path": RESULTS / "objectives/guacamol_transformer_chelator_lastblock_final/analysis/distribution_distance_fcd_10seed/distribution_distance_metrics.csv",
        "models": {
            "base": "Base",
            "random_neon_last_block_output_lambda_1.0": "Random NE 1.0",
            "positive": "Positive FT",
            "neon_last_block_output_lambda_1.0": "NE 1.0",
        },
    },
}
SHIFT_LABEL_ORDER = ["Base", "Random NE 1.0", "Positive FT", "NE 1.0"]
SHIFT_COLORS = {
    "Base": METHOD_COLORS["base"],
    "Random NE 1.0": "#A3A3A3",
    "Positive FT": METHOD_COLORS["positive"],
    "NE 1.0": NE_LAMBDA_COLORS[1.00],
}

shift_frames = []
for generator, meta in GUACAMOL_SHIFT_SOURCES.items():
    if not meta["path"].exists():
        print(f"Missing distribution-distance table: {meta['path'].relative_to(ROOT)}")
        continue
    frame = pd.read_csv(meta["path"])
    if "seed" in frame.columns:
        frame = frame[frame["seed"].isin(PAPER_SEEDS_10)].copy()
    frame = frame[frame["reference"].eq("base") & frame["model"].isin(meta["models"])].copy()
    frame["generator"] = generator
    frame["method_label"] = frame["model"].map(meta["models"])
    shift_frames.append(frame)

shift_data = pd.concat(shift_frames, ignore_index=True) if shift_frames else pd.DataFrame()
if not shift_data.empty:
    shift_data["method_label"] = pd.Categorical(shift_data["method_label"], SHIFT_LABEL_ORDER, ordered=True)

fig, axes = plt.subplots(2, 2, figsize=(9.0, 6.0), constrained_layout=True, sharex=True)
for row, generator in enumerate(GUACAMOL_SHIFT_SOURCES):
    panel = shift_data[shift_data["generator"].eq(generator)] if not shift_data.empty else pd.DataFrame()
    for col, metric in enumerate(["fcd", "fdd"]):
        ax = axes[row, col]
        if not panel.empty and metric in panel.columns:
            sns.boxplot(
                data=panel,
                x="method_label",
                y=metric,
                order=SHIFT_LABEL_ORDER,
                palette=[SHIFT_COLORS[label] for label in SHIFT_LABEL_ORDER],
                width=0.55,
                linewidth=0.5,
                fliersize=0,
                ax=ax,
            )
            ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
        else:
            ax.text(0.5, 0.5, "Missing data", ha="center", va="center")
        ax.set_xlabel("")
        ax.set_ylabel(("FCD" if metric == "fcd" else "FDD") if col == 0 else ("FCD" if metric == "fcd" else "FDD"))
        if row == 0:
            ax.set_title("FCD to base" if metric == "fcd" else "FDD to base", loc="left", fontweight="bold")
        if col == 0:
            ax.text(-0.28, 0.5, generator, transform=ax.transAxes, rotation=90, va="center", ha="center", fontweight="bold")
        style_ax(ax)

label_subplots(fig, x=-0.14, y=1.04)
save_figure(fig, "si_guacamol_actual_fcd_fdd_shifts")
plt.show()

if not shift_data.empty:
    shift_data.to_csv(DATA_DIR / "si_guacamol_actual_fcd_fdd_shifts.csv", index=False)


## Statistical Summaries Supporting Main Figures

These sections collect the paired endpoint statistics generated during figure construction. They are intended for SI tables/plots, not for replacing the main-panel effect-size plots.


In [ ]:
STATS_DIR = ROOT / "notebooks" / "figures" / "plotting_data"
STAT_FILES = {
    "GuacaMol QED endpoint tests": STATS_DIR / "guacamol_qed" / "guacamol_qed_endpoint_paired_t_stats.csv",
    "GuacaMol liability endpoint tests": STATS_DIR / "guacamol_liability" / "guacamol_liability_endpoint_paired_t_stats.csv",
    "REINVENT liability endpoint tests": STATS_DIR / "reinvent_liability" / "reinvent_liability_neon1_paired_t_stats.csv",
}

stats_tables = {}
for name, path in STAT_FILES.items():
    if path.exists():
        stats_tables[name] = pd.read_csv(path)
        print(f"Loaded {name}: {path.relative_to(ROOT)} ({len(stats_tables[name])} rows)")
    else:
        print(f"Missing {name}: {path.relative_to(ROOT)}")


### Liability-Erasure Endpoint Statistics Plan

For RNN, Transformer, and REINVENT liability-erasure experiments, we test two endpoint outcomes:

1. **Target liability hit fraction**: lower is better and tests whether the intended structural liability was removed.
2. **Valid, unique, liability-free yield**: higher is better and tests whether removal translates into more retained molecules per fixed sampling budget. REINVENT runs use the exact `usable_yield` column. Older GuacaMol RNN/Transformer runs did not store the novelty-aware intersection per molecule, so the SI statistic reconstructs a fixed-budget `valid_unique_liability_free_yield` from the sample files and labels it explicitly.

Comparisons are paired by seed and use NE as the focal method: NE vs Base, NE vs Random NE, and NE vs Positive FT. We report paired mean differences, 95% t confidence intervals, paired t-test p-values, and Holm-adjusted p-values across the liability endpoint family.


In [ ]:
LIABILITY_COMPARISONS = [
    ("NE - Base", "base", "Base", METHOD_COLORS["base"]),
    ("NE - Random NE", "random_ne", "Random NE", "#A3A3A3"),
    ("NE - Positive FT", "positive", "Positive FT", METHOD_COLORS["positive"]),
]

LIABILITY_GENERATORS = {
    "GuacaMol RNN": {
        "root": RESULTS / "objectives",
        "dir_key": "rnn_dir",
        "models": {"base": "base", "positive": "positive", "random_ne": "random_neon_lambda_1.0", "ne": "neon_lambda_1.0"},
        "yield_kind": "valid_unique_liability_free_yield",
    },
    "GuacaMol Transformer": {
        "root": RESULTS / "objectives",
        "dir_key": "transformer_dir",
        "models": {"base": "base", "positive": "positive", "random_ne": "random_neon_last_block_output_lambda_1.0", "ne": "neon_last_block_output_lambda_1.0"},
        "yield_kind": "valid_unique_liability_free_yield",
    },
    "REINVENT prior": {
        "root": RESULTS / "external/reinvent4",
        "dir_key": "reinvent_dir",
        "models": {"base": "base", "positive": "positive", "random_ne": "random_neon_lambda_1", "ne": "neon_lambda_1"},
        "yield_kind": "usable_yield",
    },
}

def fixed_budget_valid_unique_liability_free_yield(seed_dir: Path, model_name: str, liability_sample_col: str) -> float:
    sample_path = seed_dir / f"{model_name}_samples.csv"
    if not sample_path.exists():
        return np.nan
    frame = pd.read_csv(sample_path)
    if frame.empty or "valid" not in frame.columns or "canonical_smiles" not in frame.columns or liability_sample_col not in frame.columns:
        return np.nan
    valid = frame["valid"].fillna(False).astype(bool)
    liability_free = frame[liability_sample_col].fillna(1).astype(float).eq(0)
    usable = frame.loc[valid & liability_free, "canonical_smiles"].dropna().astype(str).nunique()
    return usable / len(frame)

def load_liability_endpoint_frame() -> pd.DataFrame:
    rows = []
    for generator, generator_meta in LIABILITY_GENERATORS.items():
        for objective, objective_meta in plot_loaders.OBJECTIVES.items():
            result_dir = generator_meta["root"] / objective_meta[generator_meta["dir_key"]]
            metrics_path = result_dir / "objective_metrics.csv"
            if not metrics_path.exists():
                print(f"Missing metrics: {metrics_path.relative_to(ROOT)}")
                continue
            metrics = pd.read_csv(metrics_path)
            metrics = metrics[metrics["seed"].isin(PAPER_SEEDS_10)].copy()
            target_metric = objective_meta["metric"]
            liability_sample_col = target_metric.removesuffix("_fraction")
            for method_key, model_name in generator_meta["models"].items():
                subset = metrics[metrics["model"].astype(str).eq(model_name)].copy()
                if subset.empty:
                    print(f"Missing model={model_name} in {metrics_path.relative_to(ROOT)}")
                    continue
                for _, row in subset.iterrows():
                    seed = int(row["seed"])
                    usable_yield = row.get("usable_yield", np.nan)
                    yield_kind = generator_meta["yield_kind"]
                    if pd.isna(usable_yield) or yield_kind != "usable_yield":
                        usable_yield = fixed_budget_valid_unique_liability_free_yield(
                            result_dir / f"seed_{seed}",
                            model_name,
                            liability_sample_col,
                        )
                    rows.append({
                        "generator": generator,
                        "objective": objective,
                        "objective_label": objective_meta["label"],
                        "seed": seed,
                        "method_key": method_key,
                        "model": model_name,
                        "target_hit_fraction": row[target_metric],
                        "usable_output_yield": usable_yield,
                        "yield_kind": yield_kind,
                    })
    return pd.DataFrame(rows)

def holm_adjust(p_values: pd.Series) -> pd.Series:
    values = p_values.astype(float).to_numpy()
    adjusted = np.full(len(values), np.nan)
    valid = np.where(np.isfinite(values))[0]
    if len(valid) == 0:
        return pd.Series(adjusted, index=p_values.index)
    order = valid[np.argsort(values[valid])]
    running_max = 0.0
    m = len(order)
    for rank, idx in enumerate(order):
        adj = min((m - rank) * values[idx], 1.0)
        running_max = max(running_max, adj)
        adjusted[idx] = running_max
    return pd.Series(adjusted, index=p_values.index)

def paired_liability_stats(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for metric, metric_label, better in [
        ("target_hit_fraction", "Target liability hit fraction", "lower"),
        ("usable_output_yield", "Valid, unique, liability-free yield", "higher"),
    ]:
        for (generator, objective, objective_label), group in data.groupby(["generator", "objective", "objective_label"], observed=True):
            pivot = group.pivot_table(index="seed", columns="method_key", values=metric, aggfunc="first")
            if "ne" not in pivot.columns:
                continue
            for comparison, comparator_key, comparator_label, color in LIABILITY_COMPARISONS:
                if comparator_key not in pivot.columns:
                    continue
                paired = pivot[["ne", comparator_key]].dropna()
                diff = paired["ne"] - paired[comparator_key]
                if len(diff) < 2:
                    p_value = np.nan
                    ci_low = np.nan
                    ci_high = np.nan
                    std = np.nan
                else:
                    test = stats.ttest_1samp(diff, 0.0, nan_policy="omit")
                    p_value = float(test.pvalue)
                    std = float(diff.std(ddof=1))
                    sem = std / np.sqrt(len(diff))
                    margin = float(stats.t.ppf(0.975, len(diff) - 1) * sem)
                    ci_low = float(diff.mean() - margin)
                    ci_high = float(diff.mean() + margin)
                rows.append({
                    "generator": generator,
                    "objective": objective,
                    "objective_label": objective_label,
                    "metric": metric,
                    "metric_label": metric_label,
                    "better": better,
                    "comparison": comparison,
                    "comparator": comparator_key,
                    "comparator_label": comparator_label,
                    "n_pairs": int(len(diff)),
                    "ne_mean": float(paired["ne"].mean()) if len(paired) else np.nan,
                    "comparator_mean": float(paired[comparator_key].mean()) if len(paired) else np.nan,
                    "mean_difference": float(diff.mean()) if len(diff) else np.nan,
                    "std_difference": std,
                    "ci95_low": ci_low,
                    "ci95_high": ci_high,
                    "paired_t_p": p_value,
                })
    out = pd.DataFrame(rows)
    if not out.empty:
        out["holm_p"] = holm_adjust(out["paired_t_p"])
    return out

# Inferential results are computed once by scripts/analyze_paper_statistics.py.
statistics_path = RESULTS / "paper_statistics/confirmatory_paired_comparisons.csv"
if not statistics_path.exists():
    raise FileNotFoundError(f"Run `make paper-statistics` first: {statistics_path}")
liability_stats = pd.read_csv(statistics_path)
liability_stats = liability_stats[
    liability_stats["experiment"].eq("liability")
    & liability_stats["comparator"].isin(["base", "random_ne", "positive"])
].copy()
liability_stats["comparison"] = liability_stats["comparator"].map({
    "base": "NE - Base",
    "random_ne": "NE - Random NE",
    "positive": "NE - Positive FT",
})
print(f"Exact sign-flip comparison rows: {len(liability_stats):,}")
liability_stats.head()


In [ ]:
GENERATOR_ORDER = ["GuacaMol RNN", "GuacaMol Transformer", "REINVENT prior"]
METRIC_ORDER = ["target_hit_fraction", "usable_yield"]
COMPARISON_ORDER = [name for name, *_ in LIABILITY_COMPARISONS]
COMPARISON_COLORS = {name: color for name, _, _, color in LIABILITY_COMPARISONS}
OBJECTIVE_ORDER = ["Reactive", "Metal-binding motif", "Charged motif", "Assay interference"]

fig, axes = plt.subplots(len(GENERATOR_ORDER), len(METRIC_ORDER), figsize=(12.2, 8.2), constrained_layout=True, sharex=False)
for row_idx, generator in enumerate(GENERATOR_ORDER):
    for col_idx, metric in enumerate(METRIC_ORDER):
        ax = axes[row_idx, col_idx]
        panel = liability_stats[(liability_stats["generator"].eq(generator)) & (liability_stats["metric"].eq(metric))].copy()
        panel["objective_label"] = pd.Categorical(panel["objective_label"], OBJECTIVE_ORDER, ordered=True)
        panel["comparison"] = pd.Categorical(panel["comparison"], COMPARISON_ORDER, ordered=True)
        panel = panel.sort_values(["objective_label", "comparison"])
        y_base = {objective: i for i, objective in enumerate(OBJECTIVE_ORDER)}
        offsets = {"NE - Base": -0.22, "NE - Random NE": 0.0, "NE - Positive FT": 0.22}
        for comparison in COMPARISON_ORDER:
            part = panel[panel["comparison"].astype(str).eq(comparison)]
            if part.empty:
                continue
            y = [y_base[obj] + offsets[comparison] for obj in part["objective_label"].astype(str)]
            x = 100.0 * part["mean_difference"].to_numpy(dtype=float)
            xerr = np.vstack([x - 100.0 * part["ci95_low"].to_numpy(dtype=float), 100.0 * part["ci95_high"].to_numpy(dtype=float) - x])
            ax.errorbar(
                x,
                y,
                xerr=xerr,
                fmt="o",
                color=COMPARISON_COLORS[comparison],
                ecolor=COMPARISON_COLORS[comparison],
                elinewidth=1.0,
                capsize=2.5,
                markersize=4.0,
                label=comparison if row_idx == 0 and col_idx == 0 else None,
            )
        ax.axvline(0, color="#111111", lw=0.8, ls="--")
        ax.set_yticks(range(len(OBJECTIVE_ORDER)))
        ax.set_yticklabels(OBJECTIVE_ORDER if col_idx == 0 else [])
        if row_idx == 0:
            title = "Target liability reduction" if metric == "target_hit_fraction" else "Fixed-budget usable yield"
            ax.set_title(title, loc="left", fontweight="bold")
        if col_idx == 0:
            ax.set_ylabel(generator)
            ax.set_xlabel("NE minus comparator\n(percentage points; lower is better)" if row_idx == len(GENERATOR_ORDER) - 1 else "")
        else:
            ax.set_ylabel("")
            ax.set_xlabel("NE minus comparator\n(percentage points; higher is better)" if row_idx == len(GENERATOR_ORDER) - 1 else "")
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        style_ax(ax)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.035))
label_subplots(fig, x=-0.16, y=1.04)
save_figure(fig, "si_confirmatory_liability_effects")
plt.show()


### Exact liability statistics tables

The p-value heatmap has been removed because color-coded transformed p-values obscure effect direction and magnitude. The forest plot loads the authoritative results from `results/paper_statistics/confirmatory_paired_comparisons.csv`, generated with `make paper-statistics`. It shows paired mean differences and 95% Student-$t$ confidence intervals; exact sign-flip and Holm-adjusted p-values are reported in the generated Supplementary Tables.


In [ ]:
# Export all loaded statistics tables into the SI plotting-data folder for traceability.
for name, table in stats_tables.items():
    safe = name.lower().replace(":", "").replace(" ", "_").replace("-", "_")
    table.to_csv(DATA_DIR / f"{safe}.csv", index=False)


## SI Figure: SemlaFlow Exploratory Parameter Selection

All panels are exploratory screens on development seed 11 documenting configuration selection, not inferential evidence. Hyperparameter development used the oxygen–nitrogen single-bond proxy objective; transfer to the four medicinal-chemistry liability families was checked before the replicated joint-objective experiment. Panels c and d show the original unnormalized random-correction diagnostic, in which the negative-extrapolation delta is combined with a fine-tuning delta learned from randomly sampled molecules. The confirmatory random controls are globally norm-matched, and positive correction is evaluated in the replicated joint-objective experiment.


In [ ]:
SEMLA_PILOT_ROOT = RESULTS / "development" / "semlaflow_parameter_selection"
SEMLA_PILOT_PATHS = {
    "diagnostic": SEMLA_PILOT_ROOT / "bad_direction_diagnostic.csv",
    "scope": SEMLA_PILOT_ROOT / "scope_screen.csv",
    "lambda": SEMLA_PILOT_ROOT / "lambda_screen.csv",
    "transfer": SEMLA_PILOT_ROOT / "liability_transfer.csv",
}
missing = [str(path) for path in SEMLA_PILOT_PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing SemlaFlow pilot summaries:\n" + "\n".join(missing))

semla_diagnostic = pd.read_csv(SEMLA_PILOT_PATHS["diagnostic"])
semla_scope = pd.read_csv(SEMLA_PILOT_PATHS["scope"])
semla_lambda = pd.read_csv(SEMLA_PILOT_PATHS["lambda"])
semla_transfer = pd.read_csv(SEMLA_PILOT_PATHS["transfer"])

SEMLA_SCOPE_LABELS = {
    "full_model": "Full model",
    "categorical_heads": "Categorical heads",
    "last_block_chemistry_tail": "Last block + chemistry tail",
}
SEMLA_SCOPE_COLORS = {
    "full_model": "#1A759F",
    "categorical_heads": "#99D98C",
    "last_block_chemistry_tail": "#52B69A",
}
SEMLA_TRANSFER_ORDER = [
    "reactive_hit",
    "chelator_hit",
    "charged_motif_hit",
    "assay_interference_hit",
]
SEMLA_TRANSFER_LABELS = {
    "reactive_hit": "Reactive",
    "chelator_hit": "Metal-binding motif",
    "charged_motif_hit": "Charged motif",
    "assay_interference_hit": "Assay interference",
}
SEMLA_PILOT_METHODS = {
    "base": ("Base", METHOD_COLORS["base"]),
    "full_model_neon_lambda_2p5": ("Standard NE 2.5", NE_LAMBDA_COLORS[0.50]),
    "full_model_random_corrected_neon_lambda_2p5": ("Random-corrected NE 2.5", "#52B69A"),
}

def parse_semla_lambda(model: str) -> float:
    return float(model.rsplit("_lambda_", 1)[1].replace("p", "."))

fig, axes = plt.subplots(2, 2, figsize=(10.2, 7.8), constrained_layout=True)

# a: bad-direction strength.
ax = axes[0, 0]
heat = (
    semla_diagnostic
    .pivot(index="learning_rate", columns="bad_epochs", values="bad_hit_fraction")
    .sort_index()
    .sort_index(axis=1)
)
heat_percent = 100.0 * heat
sns.heatmap(
    heat_percent,
    annot=heat_percent.map(lambda value: f"{value:.1f}%"),
    fmt="",
    cmap=sns.light_palette("#1A759F", as_cmap=True),
    cbar_kws={"label": "Bad-model liability-hit fraction (%)"},
    linewidths=0.8,
    linecolor="white",
    ax=ax,
)
ax.set_xlabel("Fine-tuning epochs")
ax.set_ylabel("Learning rate")
ax.set_yticklabels([f"{value:.0e}" for value in heat.index], rotation=0)
ax.set_title("Bad-direction strength", loc="left", fontweight="bold")
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#111111")
    spine.set_linewidth(0.9)

# b: standard-NE scope screen.
ax = axes[0, 1]
scope_rows = []
for _, row in semla_scope.iterrows():
    model = str(row["model"])
    if "random_corrected" in model or "_neon_lambda_" not in model:
        continue
    for scope in SEMLA_SCOPE_LABELS:
        if model.startswith(f"{scope}_neon_lambda_"):
            scope_rows.append({
                "scope": scope,
                "lambda": parse_semla_lambda(model),
                "valid_fraction": row["valid_fraction"],
                "objective_hit_fraction": row["objective_hit_fraction"],
            })
scope_plot = pd.DataFrame(scope_rows)
for scope in SEMLA_SCOPE_LABELS:
    part = scope_plot[scope_plot["scope"].eq(scope)].sort_values("lambda")
    ax.plot(
        100 * part["valid_fraction"],
        100 * part["objective_hit_fraction"],
        marker="o",
        linewidth=1.7,
        markersize=4,
        color=SEMLA_SCOPE_COLORS[scope],
        label=SEMLA_SCOPE_LABELS[scope],
    )
    for _, row in part.iterrows():
        if row["lambda"] in (0.25, 1.0, 1.5):
            ax.annotate(
                f'{row["lambda"]:g}',
                (100 * row["valid_fraction"], 100 * row["objective_hit_fraction"]),
                xytext=(3, 3),
                textcoords="offset points",
                fontsize=6.5,
                color=SEMLA_SCOPE_COLORS[scope],
            )
base_scope = semla_scope[semla_scope["model"].eq("base")].iloc[0]
ax.scatter(
    100 * base_scope["valid_fraction"],
    100 * base_scope["objective_hit_fraction"],
    marker="X",
    s=40,
    color=METHOD_COLORS["base"],
    label="Base",
    zorder=5,
)
ax.set_xlabel("Valid output yield (%)")
ax.set_ylabel("Liability-hit fraction (%)")
ax.set_title("Parameter-scope trade-off", loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=7)
style_ax(ax)

# c: full-model lambda trajectory.
ax = axes[1, 0]
lambda_rows = []
for _, row in semla_lambda.iterrows():
    model = str(row["model"])
    if model.startswith("full_model_neon_lambda_"):
        family = "Standard NE"
    elif model.startswith("full_model_random_corrected_neon_lambda_"):
        family = "Random-corrected NE"
    else:
        continue
    lambda_rows.append({
        "family": family,
        "lambda": parse_semla_lambda(model),
        "valid_fraction": row["valid_fraction"],
        "objective_hit_fraction": row["objective_hit_fraction"],
    })
lambda_plot = pd.DataFrame(lambda_rows)
family_colors = {"Standard NE": NE_LAMBDA_COLORS[0.50], "Random-corrected NE": "#52B69A"}
for family in ("Standard NE", "Random-corrected NE"):
    part = lambda_plot[lambda_plot["family"].eq(family)].sort_values("lambda")
    ax.plot(
        100 * part["valid_fraction"],
        100 * part["objective_hit_fraction"],
        marker="o",
        linewidth=1.8,
        markersize=4,
        color=family_colors[family],
        label=family,
    )
    for _, row in part.iterrows():
        ax.annotate(
            f'{row["lambda"]:g}',
            (100 * row["valid_fraction"], 100 * row["objective_hit_fraction"]),
            xytext=(3, 2),
            textcoords="offset points",
            fontsize=6.2,
            color=family_colors[family],
        )
base_lambda = semla_lambda[semla_lambda["model"].eq("base")].iloc[0]
ax.scatter(
    100 * base_lambda["valid_fraction"],
    100 * base_lambda["objective_hit_fraction"],
    marker="X",
    s=40,
    color=METHOD_COLORS["base"],
    label="Base",
    zorder=5,
)
ax.set_xlabel("Valid output yield (%)")
ax.set_ylabel("Liability-hit fraction (%)")
ax.set_title("Random correction rescues the lambda trajectory", loc="left", fontweight="bold")
ax.legend(frameon=False)
style_ax(ax)

# d: transfer to the four paper liability families.
ax = axes[1, 1]
x = np.arange(len(SEMLA_TRANSFER_ORDER), dtype=float)
width = 0.24
for offset_index, (model, (label, color)) in enumerate(SEMLA_PILOT_METHODS.items()):
    part = (
        semla_transfer[semla_transfer["model"].eq(model)]
        .set_index("objective")
        .reindex(SEMLA_TRANSFER_ORDER)
    )
    ax.bar(
        x + (offset_index - 1) * width,
        100 * part["objective_hit_fraction"],
        width=width,
        color=color,
        edgecolor="#111111",
        linewidth=0.7,
        label=label,
    )
ax.set_xticks(
    x,
    [SEMLA_TRANSFER_LABELS[objective] for objective in SEMLA_TRANSFER_ORDER],
    rotation=25,
    ha="right",
)
ax.set_ylabel("Liability-hit fraction (%)")
ax.set_title("Transfer across liability families", loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=7)
style_ax(ax)

for panel_label, panel_ax in zip("abcd", axes.flat):
    panel_ax.text(-0.13, 1.04, panel_label, transform=panel_ax.transAxes, fontsize=11, fontweight="bold", va="bottom")
save_figure(fig, "si_semlaflow_parameter_selection")
plt.show()

semla_diagnostic.to_csv(DATA_DIR / "si_semlaflow_bad_direction_screen.csv", index=False)
scope_plot.to_csv(DATA_DIR / "si_semlaflow_scope_screen.csv", index=False)
lambda_plot.to_csv(DATA_DIR / "si_semlaflow_lambda_screen.csv", index=False)
semla_transfer.to_csv(DATA_DIR / "si_semlaflow_liability_transfer.csv", index=False)


## SI Figure: Fine-tuning Epoch Sensitivity

Development-seed analysis of reactive-liability removal for the GuacaMol RNN, GuacaMol Transformer, and REINVENT4 prior. The figure jointly audits liability removal, validity, and fixed-budget unique usable scaffold yield. It is a sensitivity analysis rather than a confirmatory multi-seed comparison. Generate the input tables with `scripts/analyze_epoch_sensitivity_scaffolds.py`.


In [ ]:
EPOCH_SENSITIVITY_ROOTS = {
    "RNN": RESULTS / "development/guacamol_rnn_reactive_epoch_sensitivity_seed_11",
    "Transformer": RESULTS / "development/guacamol_transformer_reactive_epoch_sensitivity_seed_11",
    "REINVENT4": RESULTS / "development/reinvent_reactive_epoch_sensitivity_seed_11",
}

epoch_frames = []
for architecture, result_dir in EPOCH_SENSITIVITY_ROOTS.items():
    endpoints = pd.read_csv(result_dir / "epoch_sensitivity_summary.csv")
    scaffolds = pd.read_csv(result_dir / "usable_scaffold_metrics.csv")
    merged = endpoints.merge(
        scaffolds[[
            "model",
            "n_unique_usable_scaffolds",
            "unique_usable_scaffold_yield",
            "unique_scaffold_fraction_among_usable",
            "top10_usable_scaffold_fraction",
        ]],
        on="model",
        how="inner",
        validate="one_to_one",
    )
    merged["architecture"] = architecture
    merged["direction_kind"] = merged["direction_kind"].replace({
        "raw_ne": "standard_ne",
        "norm_matched_ne": "epoch_norm_ne",
    })
    merged["valid_output_yield"] = merged["n_valid"] / merged["n_sampled"]
    epoch_frames.append(merged)
epoch_sensitivity = pd.concat(epoch_frames, ignore_index=True)

EPOCH_METHODS = {
    "positive_finetune": ("Positive FT", METHOD_COLORS["positive"]),
    "standard_ne": ("Standard NE", NE_LAMBDA_COLORS[0.50]),
    "epoch_norm_ne": ("Epoch-norm NE", "#52B69A"),
}
EPOCH_PANELS = [
    ("reactive_hit_fraction", "Reactive-liability hits"),
    ("valid_output_yield", "Valid output yield"),
    ("unique_usable_scaffold_yield", "Usable scaffold yield"),
]

fig, axes = plt.subplots(3, 3, figsize=(7.2, 7.7), sharex=True)
for row_index, architecture in enumerate(EPOCH_SENSITIVITY_ROOTS):
    architecture_data = epoch_sensitivity[epoch_sensitivity["architecture"].eq(architecture)]
    base_row = architecture_data[architecture_data["direction_kind"].eq("base")].iloc[0]
    for column_index, (metric, title) in enumerate(EPOCH_PANELS):
        ax = axes[row_index, column_index]
        ax.axhline(
            100 * base_row[metric],
            color=METHOD_COLORS["base"],
            linestyle="--",
            linewidth=1.3,
            label="Base",
            zorder=1,
        )
        for direction_kind, (label, color) in EPOCH_METHODS.items():
            part = (
                architecture_data[architecture_data["direction_kind"].eq(direction_kind)]
                .sort_values("epoch")
            )
            ax.plot(
                part["epoch"],
                100 * part[metric],
                marker="o",
                markersize=3.8,
                linewidth=1.6,
                color=color,
                label=label,
                zorder=2,
            )
        ax.set_xticks([1, 2, 5, 10])
        if architecture == "REINVENT4" and metric == "valid_output_yield":
            ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
            ax.yaxis.set_major_formatter(PercentFormatter(100, decimals=1))
        else:
            ax.yaxis.set_major_formatter(PercentFormatter(100, decimals=0))
        if row_index == 0:
            ax.set_title(title, loc="left", fontweight="bold")
        if column_index == 0:
            ax.set_ylabel(f"{architecture}\nFraction (%)")
        if row_index == len(EPOCH_SENSITIVITY_ROOTS) - 1:
            ax.set_xlabel("Fine-tuning epochs")
        style_ax(ax)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.01), ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.945), h_pad=3.25, w_pad=1.15)
label_subplots(fig, labels=list("abcdefghi"), x=-0.16, y=1.035)
save_figure(fig, "si_generator_epoch_sensitivity")
plt.show()

epoch_sensitivity.to_csv(DATA_DIR / "si_generator_epoch_sensitivity.csv", index=False)
